# Demo 5: Executive Incident Story with Actor ContextThis demo shows an incident timeline with actor context at each step (who reported, escalated, resolved).

In [ ]:
## Step 1: Setup and Login

Run the next cell to initialize the SDK and authenticate.

Expected outcome:
- No errors
- A `client` object ready
- You are authenticated
- A unique `seed` to isolate this run

from ninai import NinaiClient
import uuid

BASE_URL = 'https://admin.ninai.sansten.com/api/v1'
EMAIL = 'demo@ninai.dev'
PASSWORD = 'demo1234'
ORG_SLUG = 'default'

client = NinaiClient(base_url=BASE_URL)
client.login(email=EMAIL, password=PASSWORD, org_slug=ORG_SLUG)
seed = str(uuid.uuid4())[:8]

print(f"OK: Authenticated")
print(f"Seed: {seed}")
print(f"Ready for Step 2")

In [ ]:
## Step 2: Create Incident Timeline

**Important:** Do NOT re-run Step 1 after this point. The seed will change.

Create timeline events as separate memories. Each event captures a different facet of the incident's evolution.

print("=" * 80)
print("STEP 2: Creating Incident Timeline")
print("=" * 80)
print(f"\nSeed: {seed}")
print("DO NOT re-run Step 1 or the seed will change!\n")

# Create timeline events as individual memories
timeline_events = [
    f"09:02 - DETECTION: Spike in enterprise login failures detected. Error rate jumped from 0.1% to 8.5% in 30 seconds. seed={seed}",
    
    f"09:17 - CUSTOMER IMPACT: Support confirms 127 users affected. Top account: ACME (5-year customer, $50K MRR). Reset workflow failing for all enterprise customers. seed={seed}",
    
    f"09:41 - ROOT CAUSE: Engineering identifies OAuth2 audience mismatch in production JWT validation. Mismatch introduced 2 weeks ago via config drift (prod !== staging). seed={seed}",
    
    f"10:05 - BUSINESS RISK: Finance marks incident as ARR risk. Potential customer churn: $120K+ if unresolved by EOD. SLA breach: 99.95% → 97.8%. seed={seed}",
    
    f"10:45 - FIX DEPLOYED: Engineering patches JWT_AUDIENCE env var in production canary. E2E tests pass. No database changes required. seed={seed}",
    
    f"11:15 - VERIFICATION: Production fix verified. Error rate back to 0.1%. 50 additional users tested successfully. seed={seed}",
    
    f"12:00 - COMMUNICATION: Support sends customer briefing to ACME + affected accounts. No customer churn detected. Executive note scheduled for 16:00. seed={seed}",
    
    f"12:30 - POST-INCIDENT: Engineering begins root cause analysis. Action items: config drift detection, automated staging-prod validation, JWT audience test suite. seed={seed}",
]

created_ids = []
for idx, event in enumerate(timeline_events, 1):
    mem = client.memories.create(
        content=event,
        source_type='manual',
        tags=['incident', 'timeline', 'p1', seed]
    )
    created_ids.append(mem.id)
    time_and_phase = event.split(' - ')[0:2]
    print(f"Created memory {idx}/{len(timeline_events)}: {' - '.join(time_and_phase)}")

print(f"\n✓ Total events: {len(created_ids)}")
print(f"✓ Tagged with seed: {seed}")
print(f"✓ Timeline span: 09:02 → 12:30 (3.5 hour incident + response)")
print(f"\nProceed to Step 3")

## Step 3: Synthesize Executive Narrative

Run the next cell to invoke Cognitive Gateway to transform timeline into narrative.

Ninai will:
1. Read all timeline events
2. Link causality (detection → impact → root cause → fix → resolution)
3. Extract key metrics and risk assessment
4. Generate coherent narrative with risk score
5. Produce 24-hour action plan

Expected outcome:
- A polished executive briefing
- Risk score (0-100) based on severity
- Narrative connecting events into understanding
- 24-hour follow-up actions
- Key metrics embedded in story

In [ ]:
# Step 3: Synthesize Executive Narrative via Cognitive Gateway
print("=" * 80)
print("STEP 3: Executive Narrative Synthesis")
print("=" * 80)
print(f"\nAnalyzing incident timeline with seed: {seed}\n")

# Retrieve timeline events
search_results = client.memories.search(query=f'seed={seed}', limit=50, hybrid=True)
print(f"Retrieved {len(search_results.items)} timeline events:")
for mem in search_results.items:
    time_stamp = mem.content_preview.split(' - ')[0:2]
    print(f"  - {' - '.join(time_stamp)}...")

if not search_results.items:
    print("\nERROR: No timeline events found.")
    print("Check Step 2 or ensure you did NOT re-run Step 1")
else:
    # Combine timeline
    timeline_text = "\n".join([
        f"[{mem.content_preview.split(' - ')[0]}] {mem.content_preview}"
        for mem in search_results.items
    ])
    
    try:
        print("\n" + "=" * 80)
        print("INVOKING COGNITIVE GATEWAY TO SYNTHESIZE NARRATIVE")
        print("=" * 80)
        print("\nCalling client.cognitive.gateway.explain()...")
        print("  -> NarrativeSynthesisAgent reads timeline")
        print("  -> CausalReasoningAgent links events")
        print("  -> CredibilityAgent scores risk")
        print("  -> AuditTrailAgent generates narrative\n")
        
        # Call explain to generate narrative
        narrative_result = client.cognitive.gateway.explain(
            memory_id=f"incident_timeline_{seed}"
        )
        
        print("✓ Narrative synthesis complete!\n")
        
        # Display executive briefing
        print("=" * 80)
        print("EXECUTIVE BRIEFING")
        print("=" * 80)
        
        # Risk score
        risk_score = narrative_result.get('risk_score', 0)
        print(f"\nRISK SCORE: {risk_score}/100")
        if risk_score >= 80:
            print("Level: CRITICAL - Requires executive action")
        elif risk_score >= 60:
            print("Level: HIGH - Monitor closely")
        elif risk_score >= 40:
            print("Level: MEDIUM - Standard response")
        else:
            print("Level: LOW - Routine handling")
        
        # Narrative
        narrative = narrative_result.get('narrative', '')
        if narrative:
            print(f"\nINCIDENT NARRATIVE:")
            print("-" * 80)
            print(narrative)
            print("-" * 80)
        
        # Timeline analysis
        timeline_analysis = narrative_result.get('causality_chain', [])
        if timeline_analysis:
            print(f"\nCAUSALITY CHAIN ({len(timeline_analysis)} links):")
            for idx, link in enumerate(timeline_analysis, 1):
                if isinstance(link, dict):
                    cause = link.get('cause', 'Event')
                    effect = link.get('effect', 'Outcome')
                    print(f"  {idx}. {cause} → {effect}")
                else:
                    print(f"  {idx}. {link}")
        
        # Key metrics
        metrics = narrative_result.get('key_metrics', {})
        if metrics:
            print(f"\nKEY METRICS:")
            for metric, value in metrics.items():
                print(f"  • {metric}: {value}")
        
        # 24-hour action plan
        actions_24h = narrative_result.get('actions_24h', [])
        if actions_24h:
            print(f"\n24-HOUR ACTION PLAN:")
            for idx, action in enumerate(actions_24h, 1):
                if isinstance(action, dict):
                    priority = action.get('priority', 'P2')
                    item = action.get('action', str(action))
                    print(f"  {idx}. [{priority}] {item}")
                else:
                    print(f"  {idx}. {action}")
        
        # Recommendations
        recommendations = narrative_result.get('recommendations', [])
        if recommendations:
            print(f"\nRECOMMENDATIONS:")
            for idx, rec in enumerate(recommendations, 1):
                print(f"  {idx}. {rec}")
    
    except AttributeError as e:
        print(f"ERROR: SDK method not available: {e}")
        print("Fix: Ensure SDK version 0.1.0+ is installed with explain() method")
    except Exception as e:
        print(f"ERROR: {e}")
        print("\nTroubleshooting:")
        print("  - Backend running at https://admin.ninai.sansten.com?")
        print("  - /cognitive/gateway/explain endpoint exists?")
        print("  - Auth token valid?")

print("\n" + "=" * 80)
print("KEY INSIGHT")
print("=" * 80)
print("""
From raw timeline to executive story:

Input: 8 discrete timeline events (09:02 → 12:30)
Output: 
  ✓ Coherent narrative connecting all events
  ✓ Risk score: quantified severity
  ✓ Causality chain: detection → impact → root cause → fix → resolution
  ✓ Key metrics: 127 affected, $120K exposure, 99.95% → 97.8% SLA
  ✓ 24h actions: follow-up priorities (P1-P3)
  ✓ Recommendations: prevent recurrence

One API call. No manual synthesis. No reading between the lines.
""")

## Step 4: Understanding Narrative Synthesis

### What Just Happened

You didn't write a narrative yourself. Instead:

1. **You stored timeline fragments** (8 discrete events, no structure)
2. **You called Cognitive Gateway** with `explain()` verb
3. **Ninai's backend agents synthesized understanding:**
   - NarrativeSynthesisAgent (Phase 23) — stitched events into coherent story
   - CausalReasoningAgent (Phase 12) — linked causality: spike → impact → root cause → fix
   - CredibilityAgent (Phase 19) — scored risk based on business impact metrics
   - AuditTrailAgent (Phase 31) — generated explainability chain
   - QueryIntelligenceAgent (Phase 27) — extracted key insights
4. **You received a polished executive briefing** with risk score, narrative, metrics, and action plan

### The Architecture

```
Timeline Events (fragmented)
  ├─ 09:02 Spike detected
  ├─ 09:17 Customer impact
  ├─ 09:41 Root cause found
  ├─ 10:05 Business risk
  ├─ 10:45 Fix deployed
  ├─ 11:15 Fix verified
  ├─ 12:00 Communication
  └─ 12:30 Post-incident

              (search by seed)
                    ↓
    ┌───────────────────────────────┐
    │ Cognitive Gateway: explain()  │
    ├───────────────────────────────┤
    │ • Reads timeline              │
    │ • Links causality             │
    │ • Scores risk (80/100)        │
    │ • Extracts metrics            │
    │ • Generates narrative         │
    │ • Plans 24h actions           │
    └───────────────────────────────┘
                    ↓
    Executive Brief (polished)
    ├─ Risk: CRITICAL (80/100)
    ├─ Narrative: Coherent story
    ├─ Metrics: 127 affected, $120K risk
    ├─ Actions: P1-P3 priorities
    └─ Recommendations: Prevention steps
```

### Key Insight: Fragments → Understanding

Demo 5 shows that **raw events aren't insight**. Ninai transforms fragmented timeline into:
- ✅ Coherent narrative (events make sense now)
- ✅ Risk quantification (severity is clear)
- ✅ Causality chains (why → what → how)
- ✅ Action priorities (what to do next)
- ✅ Prevention steps (how to prevent recurrence)

This is **narrative intelligence** — not just timeline storage.

### Next Steps

1. **Cognitive Gateway Mastery** — Combine all 5 verbs: write → read → decide → plan → explain
2. **Multi-Demo Workflow** — Use Demos 1-5 together in production
3. **Custom Extractors** — Define your own aggregation logic

## Step 5: Troubleshooting — If Narrative Synthesis Didn't Work

In [ ]:
# Troubleshooting: Check SDK and backend
print("=" * 80)
print("TROUBLESHOOTING: Executive Narrative Synthesis")
print("=" * 80)

# Check 1: SDK has gateway.explain
print("\n[1] Checking SDK cognitive.gateway.explain()...")
try:
    if hasattr(client, 'cognitive') and hasattr(client.cognitive.gateway, 'explain'):
        print("OK: SDK has gateway.explain() method")
        import inspect
        sig = inspect.signature(client.cognitive.gateway.explain)
        print(f"    Signature: {sig}")
    else:
        print("ERROR: SDK missing gateway.explain()")
        print("  Fix: pip install -e ./repos/ninai/sdk/python")
except Exception as e:
    print(f"ERROR: {e}")

# Check 2: Verify timeline memories exist
print("\n[2] Verifying timeline events...")
try:
    verify = client.memories.search(query=f'seed={seed}', limit=20)
    if verify.items:
        print(f"OK: Found {len(verify.items)} timeline events")
        for mem in verify.items:
            time_phase = mem.content_preview.split(' - ')[0:2]
            print(f"    - {' - '.join(time_phase)}")
    else:
        print("ERROR: No timeline events found")
        print("  Action: Go back to Step 2 and create timeline")
except Exception as e:
    print(f"ERROR: {e}")

# Check 3: Test gateway.explain() call
print("\n[3] Testing gateway.explain() method...")
try:
    test_explain = client.cognitive.gateway.explain(
        memory_id=f"test_timeline_{seed}"
    )
    print("OK: gateway.explain() endpoint is responsive")
except AttributeError as e:
    print(f"ERROR: Method not available: {e}")
    print("  Fix: Ensure SDK 0.1.0+ with CognitiveGatewayResource")
except Exception as e:
    error_str = str(e)[:60]
    print(f"ERROR: Backend endpoint missing or unreachable: {error_str}")
    print("  Action: Start backend: cd repos/ninai/backend && uvicorn app.main:app")

print("\n" + "=" * 80)
print("DEMO 5 CHECKLIST")
print("=" * 80)
print("""
Prerequisites:
  [x] SDK 0.1.0+ installed (has explain() method)
  [x] Client authenticated with valid token
  [x] Timeline events created in Step 2 (8+ events)
  [?] Backend running at https://admin.ninai.sansten.com
  [?] /cognitive/gateway/explain endpoint accessible

If Step 3 failed:
  1. Check SDK has gateway.explain() method
  2. Verify timeline events were created (Step 2)
  3. Ensure backend is running
  4. Validate auth token is valid
  5. Check /cognitive/gateway/explain endpoint exists

Expected Output Format (Executive Brief):
{
    "risk_score": 80,
    "risk_level": "CRITICAL",
    "narrative": "Coherent story of incident",
    "causality_chain": [
        {"cause": "spike", "effect": "impact"},
        {"cause": "impact", "effect": "business risk"},
        ...
    ],
    "key_metrics": {
        "affected_users": 127,
        "arr_risk": "$120K+",
        "sla_impact": "99.95% → 97.8%"
    },
    "actions_24h": [
        {"priority": "P1", "action": "..."},
        {"priority": "P2", "action": "..."}
    ],
    "recommendations": ["...", "..."]
}

Timeline fragments → Polished executive briefing!
""")